# 03 — Potential Demand Model
**DataStorm v7.0 | SkyNet Team**

Goal: Estimate the 'latent demand' for each outlet by uncapping censored observations.

Approach:
1. **Ensemble Modeling**: Combine a Tobit-style regression (for censored demand) with XGBoost (for complex feature interactions).
2. **Peer-Group Benchmarking**: Use the 90th percentile of similar outlets as a sanity-check ceiling.
3. **Uplift Application**: Apply the constraint-based uplift factors identified in Notebook 02.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

GOLD_DIR = ROOT / 'data' / 'gold'
REPORTS  = ROOT / 'reports'
OUTPUTS  = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style='darkgrid')
print('Imports successful')

Imports successful


## 1. Load Data

In [3]:
df = pd.read_csv(GOLD_DIR / 'outlet_features_with_constraints.csv')
print('Data loaded:', df.shape)

# Target for the model is current observed mean volume
# We will predict this, then uplift the predictions based on constraint labels
target = 'mean_monthly_volume'

# Features to use
features = [
    'cv_volume', 'months_active', 'zero_order_months', 'peak_to_avg_ratio',
    'credit_limit_hit_rate', 'stockout_flag', 'volume_trend',
    'jan26_seasonality_index', 'outlet_size_ord', 'Cooler_Count'
]

# Add POI features if they exist
poi_cols = [c for c in df.columns if c.startswith('poi_')]
features.extend(poi_cols)

# Add one-hot encoded cols
one_hot_cols = [c for c in df.columns if 'Outlet_Size_' in c or 'Outlet_Type_' in c]
features.extend(one_hot_cols)

print(f'Using {len(features)} features')

Data loaded: (10492, 36)
Using 18 features


## 2. Model Training (Predicting Observed Demand)

In [4]:
X = df[features]
y = df[target]

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

val_preds = model.predict(X_val)
print(f'Validation R2: {r2_score(y_val, val_preds):.4f}')
print(f'Validation MAE: {mean_absolute_error(y_val, val_preds):.4f}')

Validation R2: 0.9959
Validation MAE: 11.4638


## 3. Demand Uncapping & Potential Estimation

In [5]:
# 1. Model predictions for all outlets
df['predicted_base_demand'] = model.predict(X)

# 2. Apply constraint-based uplift (Uncapping)
# Use the max of (observed mean, predicted base) as our starting point
df['potential_latent_demand'] = np.maximum(df['mean_monthly_volume'], df['predicted_base_demand']) * df['uplift_prior']

# 3. Sanity check: Cap at peer-group 90th percentile
# We don't want to predict implausible volumes even with constraints
df['final_potential_estimate'] = np.minimum(df['potential_latent_demand'], df['peer_p90_volume'] * 1.5)

print('Potential estimation summary:')
print(df[['mean_monthly_volume', 'final_potential_estimate']].describe())

Potential estimation summary:
       mean_monthly_volume  final_potential_estimate
count         10492.000000              10492.000000
mean            221.197586                199.372074
std             318.819780                235.339998
min              19.148853                 24.476093
25%              41.313266                 43.446979
50%             107.101192                119.916008
75%             157.842416                158.950615
max            1485.620219                855.924458


## 4. January 2026 Projection

In [6]:
# Apply January 2026 seasonality and holiday adjustment
# seasonality index is already factored into feature engineering gold layer

df['jan_2026_forecast'] = df['final_potential_estimate'] * df['jan26_seasonality_index']

# Holiday adjustment (heuristic: +2% for every public holiday above average)
avg_holidays = 2
hol_adj = 1 + (df['jan_total_holidays'] - avg_holidays) * 0.02
df['jan_2026_forecast'] = df['jan_2026_forecast'] * hol_adj

print('January 2026 Forecast Statistics:')
print(df['jan_2026_forecast'].describe())

January 2026 Forecast Statistics:
count    10492.000000
mean       195.355316
std        230.790971
min         23.497050
25%         42.472361
50%        115.517271
75%        160.192007
max        944.940602
Name: jan_2026_forecast, dtype: float64


## 5. Save Model State & Potential Data

In [7]:
df.to_csv(GOLD_DIR / 'potential_demand_results.csv', index=False)
print(f'Saved potential demand results to {GOLD_DIR}')

Saved potential demand results to C:\Users\User\Documents\Projects\Datastorm\SkyNet-datastorm-v7\data\gold
